## Set up

In [1]:
%pip install neo4j
%pip install matplotlib


[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 23.3.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Imports

In [1]:
from neo4j import GraphDatabase

#### Load classes

In [ ]:
%run ../commons/models.py

### Establish connection and create driver

In [23]:
uri = "bolt://0.0.0.0:7687"
username = "neo4j"
password = "111122223333"
driver = GraphDatabase.driver(uri, auth=(username, password))

#### Get the image data

In [3]:
query = """
    MATCH (target {image_id: '7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47'})-[*0..5]-(connectedNode)
    WHERE NOT "Line" IN labels(connectedNode)
    RETURN connectedNode
"""



with driver.session() as session:
    result = session.execute_read(execute_query, query)
    for record in result:
        print(record)

NameError: name 'execute_query' is not defined

#### Clean up

In [4]:
def execute_query(tx, query):
    result = tx.run(query)
    return [record for record in result]

#### Playground

In [24]:
import math


def fetch_line_data(session):
    query = """
    MATCH (line1:Line)--(:Location)--(coords1:Coordinates),
          (line2:Line)--(:Location)--(coords2:Coordinates)
    WHERE NOT EXISTS {
        MATCH (line1)-[*]-(:Line)
    }
    AND line1.id <> line2.id
    RETURN line1, coords1, line2, coords2
    """
    return session.run(query)


def calculate_angle(line1, line2):
    dx1 = line1[2] - line1[0]
    dy1 = line1[3] - line1[1]
    dx2 = line2[2] - line2[0]
    dy2 = line2[3] - line2[1]
    
    angle1 = math.atan2(dy1, dx1)
    angle2 = math.atan2(dy2, dx2)
    angle = abs(angle1 - angle2)
    
    if angle > math.pi:
        angle = 2 * math.pi - angle

    # Ensuring the angle is always the interior angle
    if angle > math.pi / 2:
        angle = math.pi - angle
    
    return math.degrees(angle)  # Convert to degrees
        
def line_intersection(line1, line2):
    x1, y1, x2, y2 = line1
    x3, y3, x4, y4 = line2

    denominator = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if denominator == 0:
        return None

    px = ((x1 * y2 - y1 * x2) * (x3 - x4) - (x1 - x2) * (x3 * y4 - y3 * x4)) / denominator
    py = ((x1 * y2 - y1 * x2) * (y3 - y4) - (y1 - y2) * (x3 * y4 - y3 * x4)) / denominator

    return px, py

def create_angle_points(session, intersection_data):
    query = """
    UNWIND $intersection_data AS data
    MERGE (ap:AnglePoint)-[:HAS]->(:AnglePointLocation {x: data.intersection.x, y: data.intersection.y, angle: data.angle})
    WITH ap, data
    MATCH (line1:Line {id: data.line1_id})
    MATCH (line2:Line {id: data.line2_id})
    MERGE (line1)-[:INTERSECTS_AT]->(ap)
    MERGE (line2)-[:INTERSECTS_AT]->(ap)
    """
    session.run(query, intersection_data=intersection_data)
    
    
with driver.session() as session:
    lines_result = fetch_line_data(session)
    intersection_data = []

    for record in lines_result:
        line1_coords = (record['coords1']['x1'], record['coords1']['y1'], record['coords1']['x2'], record['coords1']['y2'])
        line2_coords = (record['coords2']['x1'], record['coords2']['y1'], record['coords2']['x2'], record['coords2']['y2'])
        intersection = line_intersection(line1_coords, line2_coords)

        if intersection:
            angle = calculate_angle(line1_coords, line2_coords)
            intersection_data.append({
                'line1_id': record['line1']['id'],
                'line2_id': record['line2']['id'],
                'intersection': {'x': intersection[0], 'y': intersection[1]},
                'angle': angle
            })

    create_angle_points(session, intersection_data)

In [43]:
# %load ../commons/models.py
from dataclasses import dataclass
from enum import Enum

class Reason(Enum):
    FIRST_POINT = "First point"
    FIRST_LINE = "First Line"

@dataclass(frozen=True)
class CriticalPoint:
    uuid: str
    image_id: str
    reason: Reason
    connectedVerts = {}

In [46]:
def get_starting_point(results):
    critical_points = set()
    first_points = set()
    first_lines = set()
    
    for record in results:
        record = record['connectedNode']
        label = [x for x in record.labels][0]
        if label == "CriticalPoint":
            critical_point = CriticalPoint(uuid=record['uuid'], reason=record['reason'], image_id=record['image_id'])
            critical_points.add(critical_point)
            
    for value in critical_points:
        if value.reason == Reason.FIRST_POINT.value:
            first_points.add(value)
        if value.reason == Reason.FIRST_LINE.value:
            first_lines.add(value)
            
    return first_points, first_lines

In [ ]:
def comapare_nodes(node1, node2):
    if isinstance(node1, CriticalPoint):
        return compare_critical_points(node1, node2)
    else:
        return "Unsupported node type"
    
    
def compare_critical_points(cp1: CriticalPoint, cp2: CriticalPoint):
    return 1 if (cp1.reason == cp2.reason) else 0

In [49]:
first_points, first_lines = get_starting_point(result)
first_critical_points = {critical_point.image_id: critical_point for critical_point in first_points}
first_critical_lines = {critical_point.image_id: critical_point for critical_point in first_lines}
print(f"First points: {first_critical_points}")
print(f"First lines: {first_critical_lines}")

with driver.session() as session:
    go_over_contour_query = """
        MATCH (cp1:CriticalPoint {uuid: })
    """
    session.execute_read(execute_query, )

First points: {'7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47': CriticalPoint(uuid='8db2ae58-2358-43ed-a872-51e0b97750c6', image_id='7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47', reason='First point')}
First lines: {'7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47': CriticalPoint(uuid='92a27982-70b4-4aca-ba58-f07546a97cc1', image_id='7b11d7aa-e99d-4ba8-b7b1-cfd8db276d47', reason='First Line')}


#### Close the driver

In [ ]:
driver.close()